In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta

LOCALITIES_CSV = "localities_delhi_geocoded.csv"
OUT_INCIDENTS = "synthetic_wsi_delhi_localities.csv"
OUT_LOCALITY_AGG = "locality_aggregates.json"
OUT_FEATURE_ORDER = "feature_order.json"

N_ROWS = 1_000_000
RANDOM_SEED = 42

START_DATE = "2023-01-01T00:00:00"
END_DATE   = "2024-12-31T23:59:59"

SPREAD_DEG = 0.006
P_OFF_LOCALITY = 0.02   # 2% random background points

LAT_MIN, LAT_MAX = 28.40, 28.90
LON_MIN, LON_MAX = 76.90, 77.40

np.random.seed(RANDOM_SEED)

loc_df = pd.read_csv(LOCALITIES_CSV)

required_cols = {"name", "latitude", "longitude"}
if not required_cols.issubset(set(loc_df.columns)):
    raise ValueError(f"Localities CSV must contain columns: {', '.join(required_cols)}")

# Drop rows without coords
loc_df = loc_df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
if loc_df.empty:
    raise ValueError("No geocoded localities found (latitude/longitude are NaN).")

n_loc = len(loc_df)


# sample locality indices and off-locality mask 
local_indices = np.random.randint(0, n_loc, size=N_ROWS)
is_off = np.random.rand(N_ROWS) < P_OFF_LOCALITY

# pre-allocate arrays 
lat = np.empty(N_ROWS, dtype=float)
lon = np.empty(N_ROWS, dtype=float)
locality_assigned = np.empty(N_ROWS, dtype=object)

# assign coords and locality names
for i in range(N_ROWS):
    if is_off[i]:
        lat[i] = np.random.uniform(LAT_MIN, LAT_MAX)
        lon[i] = np.random.uniform(LON_MIN, LON_MAX)
        locality_assigned[i] = "UNK"
    else:
        li = local_indices[i]
        center_lat = loc_df.loc[li, "latitude"]
        center_lon = loc_df.loc[li, "longitude"]
        lat[i] = center_lat + np.random.normal(0, SPREAD_DEG)
        lon[i] = center_lon + np.random.normal(0, SPREAD_DEG)
        locality_assigned[i] = loc_df.loc[li, "name"]

# timestamps uniformly sampled 
start_ts = pd.to_datetime(START_DATE)
end_ts   = pd.to_datetime(END_DATE)
total_seconds = int((end_ts - start_ts).total_seconds())
rand_seconds = np.random.randint(0, total_seconds + 1, size=N_ROWS)
timestamps = start_ts + pd.to_timedelta(rand_seconds, unit="s")

# categorical draws 
crime_types = ["harassment","assault","theft","stalking","eve_teasing","safe_observation"]
crime_probs = np.array([0.22, 0.08, 0.30, 0.06, 0.04, 0.30])
crime_choice = np.random.choice(crime_types, size=N_ROWS, p=crime_probs)

victim_gender = np.random.choice(["female","male","other"], size=N_ROWS, p=[0.62,0.36,0.02])
offender_gender = np.random.choice(["male","female","unknown"], size=N_ROWS, p=[0.72,0.18,0.10])

# tilt female victim share for certain crimes
mask_hs = np.isin(crime_choice, ["harassment","stalking","eve_teasing"])
if mask_hs.sum() > 0:
    victim_gender[mask_hs] = np.random.choice(["female","male","other"], size=mask_hs.sum(), p=[0.82,0.17,0.01])

# severity base + locality effect + noise 
base_severity = {
    "harassment": 0.45, "assault": 0.75, "theft": 0.30,
    "stalking": 0.60, "eve_teasing": 0.40, "safe_observation": 0.05
}
severity = np.array([base_severity[c] for c in crime_choice], dtype=float)

# deterministic-ish locality risk factor
loc_risk = ((np.arange(n_loc) % 7) / 7.0) * 0.6
unk_risk = 0.25
loc_risk_map = {loc_df.loc[i, "name"]: float(loc_risk[i]) for i in range(n_loc)}

severity = severity + np.array([loc_risk_map.get(locality_assigned[i], unk_risk) for i in range(N_ROWS)])
severity = np.clip(severity + np.random.normal(0, 0.12, size=N_ROWS), 0, 1)

# lighting, crowd, poi per incident
base_lighting_per_loc = {
    loc_df.loc[i, "name"]: float(np.clip(6.0 + np.random.normal(0, 1.2), 0, 10))
    for i in range(n_loc)
}
base_crowd_per_loc = {
    loc_df.loc[i, "name"]: float(np.clip(0.45 + np.random.normal(0, 0.12), 0, 1))
    for i in range(n_loc)
}

lighting = np.empty(N_ROWS, dtype=float)
crowd = np.empty(N_ROWS, dtype=float)
poi = np.empty(N_ROWS, dtype=int)

hours = pd.DatetimeIndex(timestamps).hour

for i in range(N_ROWS):
    loc_name = locality_assigned[i]
    if loc_name == "UNK":
        base_light = 5.0 + np.random.normal(0, 1.3)
        base_crow = 0.45 + np.random.normal(0, 0.12)
        base_poi = np.random.poisson(3)
    else:
        base_light = base_lighting_per_loc.get(loc_name, 5.5)
        base_crow = base_crowd_per_loc.get(loc_name, 0.45)
        base_poi = max(0, int(np.random.poisson(5)))

    hour = hours[i]
    if 6 <= hour <= 18:
        diurnal = 0.7
    elif 19 <= hour <= 23:
        diurnal = 1.0
    else:
        diurnal = 0.45

    lighting[i] = np.clip(base_light * diurnal + np.random.normal(0, 0.6), 0, 10)
    crowd[i] = np.clip(base_crow + (0.12 if (6 <= hour <= 9 or 17 <= hour <= 22) else -0.03) + np.random.normal(0, 0.08), 0, 1)
    poi[i] = base_poi

# engineered features 
light_crowd = lighting * crowd
severity_lowlight = severity * (10.0 - lighting)
poi_to_crowd = poi / (crowd + 1e-6)

# policing index and safety calculation 
policing_per_loc = {
    loc_df.loc[i, "name"]: float(np.clip(0.5 + 0.5 * np.sin(i * 1.23), 0.05, 0.95))
    for i in range(n_loc)
}
policing_array = np.array([policing_per_loc.get(locality_assigned[i], 0.5) for i in range(N_ROWS)])

w_sev = 0.55 + 0.15 * (1 - policing_array)
w_light = 0.30 + 0.08 * (1 - policing_array)
w_crowd = 0.15 - 0.03 * (1 - policing_array)
w_sum = w_sev + w_light + w_crowd
w_sev /= w_sum; w_light /= w_sum; w_crowd /= w_sum

base_safety = 1.0 - (w_sev * severity + w_light * (1.0 - lighting / 10.0) + w_crowd * (1.0 - crowd))

sigma = 0.02 + 0.08 * severity * (1.0 - crowd)
label_noise = np.random.normal(0, sigma, size=N_ROWS)

flip_mask = (np.random.rand(N_ROWS) < 0.004)
label_shift_vals = np.random.uniform(-0.18, -0.05, size=flip_mask.sum())
label_shift_full = np.zeros(N_ROWS, dtype=float)
label_shift_full[flip_mask] = label_shift_vals

safety_index = np.clip(base_safety + label_noise + label_shift_full, 0.0, 1.0)

# assemble dataframe 
df = pd.DataFrame({
    "incident_id": np.arange(1, N_ROWS + 1),
    "timestamp": timestamps,
    "latitude": lat,
    "longitude": lon,
    "locality": locality_assigned,
    "crime_type": crime_choice,
    "victim_gender": victim_gender,
    "offender_gender": offender_gender,
    "severity_score": np.round(severity, 4),
    "lighting_score": np.round(lighting, 3),
    "crowd_density": np.round(crowd, 3),
    "poi_density": poi,
    "light_crowd": np.round(light_crowd, 3),
    "severity_lowlight": np.round(severity_lowlight, 3),
    "poi_to_crowd": np.round(poi_to_crowd, 3),
    "policing_index": np.round(policing_array, 3),
    "safety_index": np.round(safety_index, 4),
    "hour_of_day": hours,
    "is_weekend": (pd.DatetimeIndex(timestamps).dayofweek >= 5).astype(int),
    "is_night": ((hours >= 20) | (hours <= 4)).astype(int)
})

df.to_csv(OUT_INCIDENTS, index=False)
print(f"Saved synthetic incidents to: {OUT_INCIDENTS}  (rows: {len(df)})")
print("All done.")
